In [ ]:
import os
import random
import string
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt

random.seed(42)
np.random.seed(42)

In [11]:
BASE_DIR = os.path.dirname(os.getcwd())
DATA_DIR = os.path.join(BASE_DIR, 'data', 'captcha')
OUTPUT_DIR = os.path.join(BASE_DIR, 'outputs')

os.makedirs(os.path.join(DATA_DIR, 'original'), exist_ok=True)
os.makedirs(os.path.join(DATA_DIR, 'noisy'), exist_ok=True)
os.makedirs(os.path.join(DATA_DIR, 'filtered'), exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Data: {DATA_DIR}")
print(f"Outputs: {OUTPUT_DIR}")

Data: e:\University_Projects\Vision\Computer-Vision-Course\HW2\data\captcha
Outputs: e:\University_Projects\Vision\Computer-Vision-Course\HW2\outputs


In [13]:
def generate_random_text(length=4):
    characters = string.ascii_lowercase + string.digits
    return ''.join(random.choice(characters) for _ in range(length))

def create_captcha_image(text, size=(200, 80), font_size=50):
    image = Image.new('RGB', size, color='white')
    draw = ImageDraw.Draw(image)
    
    try:
        font = ImageFont.truetype("arial.ttf", font_size)
    except:
        font = ImageFont.load_default()
    
    bbox = draw.textbbox((0, 0), text, font=font)
    text_width = bbox[2] - bbox[0]
    text_height = bbox[3] - bbox[1]
    
    position = ((size[0] - text_width) // 2, (size[1] - text_height) // 2)
    
    draw.text(position, text, fill='black', font=font)
    
    return image

captcha_data = []

for i in range(1, 21):
    text = generate_random_text(4)
    
    image = create_captcha_image(text)
    
    filename = f"captcha_{i:02d}.png"
    filepath = os.path.join(DATA_DIR, 'original', filename)
    image.save(filepath)
    
    captcha_data.append({
        'id': i,
        'text': text,
        'filename': filename
    })
    
    print(f"Image {i:02d} generated: {text} -> {filename}")

print(f"\nAll 20 images were saved in {os.path.join(DATA_DIR, 'original')}")

Image 01 generated: zren -> captcha_01.png
Image 02 generated: un5z -> captcha_02.png
Image 03 generated: 3jqi -> captcha_03.png
Image 04 generated: p98q -> captcha_04.png
Image 05 generated: 1zxo -> captcha_05.png
Image 06 generated: i65f -> captcha_06.png
Image 07 generated: dhjk -> captcha_07.png
Image 08 generated: 1eyy -> captcha_08.png
Image 09 generated: 37q9 -> captcha_09.png
Image 10 generated: ah8r -> captcha_10.png
Image 11 generated: vhs1 -> captcha_11.png
Image 12 generated: k3aq -> captcha_12.png
Image 13 generated: 6l6g -> captcha_13.png
Image 14 generated: t6mj -> captcha_14.png
Image 15 generated: xk87 -> captcha_15.png
Image 16 generated: au5b -> captcha_16.png
Image 17 generated: hxtp -> captcha_17.png
Image 18 generated: dpff -> captcha_18.png
Image 19 generated: 5e8i -> captcha_19.png
Image 20 generated: i49k -> captcha_20.png

All 20 images were saved in e:\University_Projects\Vision\Computer-Vision-Course\HW2\data\captcha\original


In [14]:
df = pd.DataFrame(captcha_data)

csv_path = os.path.join(DATA_DIR, 'captcha_labels.csv')
df.to_csv(csv_path, index=False, encoding='utf-8')

df.head()

,id,text,filename
0,1,zren,captcha_01.png
1,2,un5z,captcha_02.png
2,3,3jqi,captcha_03.png
3,4,p98q,captcha_04.png
4,5,1zxo,captcha_05.png


In [ ]:
def add_salt_pepper_noise(image, salt_prob=0.03, pepper_prob=0.03):
    img_array = np.array(image)
    
    salt_mask = np.random.random(img_array.shape[:2]) < salt_prob
    img_array[salt_mask] = 255
    
    pepper_mask = np.random.random(img_array.shape[:2]) < pepper_prob
    img_array[pepper_mask] = 0
    
    return Image.fromarray(img_array)

for i in range(1, 21):
    filename = f"captcha_{i:02d}.png"
    
    original_path = os.path.join(DATA_DIR, 'original', filename)
    image = Image.open(original_path)
    
    noisy_image = add_salt_pepper_noise(image, salt_prob=0.03, pepper_prob=0.03)
    
    noisy_path = os.path.join(DATA_DIR, 'noisy', filename)
    noisy_image.save(noisy_path)
    
    print(f"Noise was added to image {i:02d}")

print(f"\nAll noisy images were saved in {os.path.join(DATA_DIR, 'noisy')}")


# Creating comparison image(outputs)
fig, axes = plt.subplots(8, 2, figsize=(10, 24))
fig.suptitle('Comparison of Original and Noisy Images', fontsize=16, y=0.995)

for idx in range(8):
    filename = f"captcha_{idx+1:02d}.png"
    
    original = Image.open(os.path.join(DATA_DIR, 'original', filename))
    noisy = Image.open(os.path.join(DATA_DIR, 'noisy', filename))
    
    axes[idx, 0].imshow(original, cmap='gray')
    axes[idx, 0].set_title(f'Original {idx+1}', fontsize=10)
    axes[idx, 0].axis('off')
    
    axes[idx, 1].imshow(noisy, cmap='gray')
    axes[idx, 1].set_title(f'Noisy {idx+1}', fontsize=10)
    axes[idx, 1].axis('off')

plt.tight_layout()
comparison_path = os.path.join(OUTPUT_DIR, 'comparison_original_noisy.png')
plt.savefig(comparison_path, dpi=150, bbox_inches='tight')
plt.close()

Noise was added to image 01
Noise was added to image 02
Noise was added to image 03
Noise was added to image 04
Noise was added to image 05
Noise was added to image 06
Noise was added to image 07
Noise was added to image 08
Noise was added to image 09
Noise was added to image 10
Noise was added to image 11
Noise was added to image 12
Noise was added to image 13
Noise was added to image 14
Noise was added to image 15
Noise was added to image 16
Noise was added to image 17
Noise was added to image 18
Noise was added to image 19
Noise was added to image 20

All noisy images were saved in e:\University_Projects\Vision\Computer-Vision-Course\HW2\data\captcha\noisy

Creating comparison image...
Comparison image saved in e:\University_Projects\Vision\Computer-Vision-Course\HW2\outputs\comparison_original_noisy.png


Explanation: Best filter for removing salt-and-pepper noise"
To remove salt-and-pepper noise, there are three main filters:

1. Median Filter (Best choice)
   - Robust against outliers
   - Preserves edges better
   - Ideal for salt-and-pepper noise

2. Mean Filter
   - Causes image blurring
   - Does not fully remove outlier noise

3. Gaussian Filter
   - Suitable for Gaussian noise
   - Less effective for salt-and-pepper noise

So we use the median filter to remove the noise.

In [ ]:
def apply_median_filter(image, size=3):
    img_array = np.array(image)
    filtered = ndimage.median_filter(img_array, size=size)
    return Image.fromarray(filtered)

def apply_mean_filter(image, size=3):
    img_array = np.array(image)
    kernel = np.ones((size, size)) / (size * size)
    
    if len(img_array.shape) == 3:
        filtered = np.zeros_like(img_array)
        for i in range(3):
            filtered[:,:,i] = ndimage.convolve(img_array[:,:,i], kernel)
    else:
        filtered = ndimage.convolve(img_array, kernel)
    
    return Image.fromarray(filtered.astype(np.uint8))

def apply_gaussian_filter(image, sigma=1):
    img_array = np.array(image)
    filtered = ndimage.gaussian_filter(img_array, sigma=sigma)
    return Image.fromarray(filtered.astype(np.uint8))

print("\nApplying median filter to noisy images...")

for i in range(1, 21):
    filename = f"captcha_{i:02d}.png"
    
    noisy_path = os.path.join(DATA_DIR, 'noisy', filename)
    noisy_image = Image.open(noisy_path)
    
    filtered_image = apply_median_filter(noisy_image, size=3)
    
    filtered_path = os.path.join(DATA_DIR, 'filtered', filename)
    filtered_image.save(filtered_path)
    
    print(f"Median filter applied to image {i:02d}")

print(f"\nAll filtered images were saved in {os.path.join(DATA_DIR, 'filtered')}")

# Creating three-stage comparison image
fig, axes = plt.subplots(6, 3, figsize=(12, 18))
fig.suptitle('Comparison: Original → Noisy → Filtered', fontsize=16, y=0.995)

for idx in range(6):
    filename = f"captcha_{idx+1:02d}.png"
    text = df[df['filename'] == filename]['text'].values[0]
    
    original = Image.open(os.path.join(DATA_DIR, 'original', filename))
    noisy = Image.open(os.path.join(DATA_DIR, 'noisy', filename))
    filtered = Image.open(os.path.join(DATA_DIR, 'filtered', filename))
    
    axes[idx, 0].imshow(original, cmap='gray')
    axes[idx, 0].set_title(f'Original: {text}', fontsize=10)
    axes[idx, 0].axis('off')
    
    axes[idx, 1].imshow(noisy, cmap='gray')
    axes[idx, 1].set_title(f'Noisy: {text}', fontsize=10)
    axes[idx, 1].axis('off')
    
    axes[idx, 2].imshow(filtered, cmap='gray')
    axes[idx, 2].set_title(f'Filtered: {text}', fontsize=10)
    axes[idx, 2].axis('off')

plt.tight_layout()
all_stages_path = os.path.join(OUTPUT_DIR, 'comparison_all_stages.png')
plt.savefig(all_stages_path, dpi=150, bbox_inches='tight')
plt.close()

# Comparing three different filters
fig, axes = plt.subplots(3, 4, figsize=(16, 10))
fig.suptitle('Comparison of Different Filters', fontsize=16, y=0.995)

for idx in range(3):
    filename = f"captcha_{idx+1:02d}.png"
    text = df[df['filename'] == filename]['text'].values[0]
    
    noisy = Image.open(os.path.join(DATA_DIR, 'noisy', filename))
    
    median_filtered = apply_median_filter(noisy, size=3)
    mean_filtered = apply_mean_filter(noisy, size=3)
    gaussian_filtered = apply_gaussian_filter(noisy, sigma=1)
    
    axes[idx, 0].imshow(noisy, cmap='gray')
    axes[idx, 0].set_title(f'Noisy: {text}', fontsize=10)
    axes[idx, 0].axis('off')
    
    axes[idx, 1].imshow(median_filtered, cmap='gray')
    axes[idx, 1].set_title(f'Median Filter ⭐', fontsize=10)
    axes[idx, 1].axis('off')
    
    axes[idx, 2].imshow(mean_filtered, cmap='gray')
    axes[idx, 2].set_title(f'Mean Filter', fontsize=10)
    axes[idx, 2].axis('off')
    
    axes[idx, 3].imshow(gaussian_filtered, cmap='gray')
    axes[idx, 3].set_title(f'Gaussian Filter', fontsize=10)
    axes[idx, 3].axis('off')

plt.tight_layout()
filter_comparison_path = os.path.join(OUTPUT_DIR, 'filter_comparison.png')
plt.savefig(filter_comparison_path, dpi=150, bbox_inches='tight')
plt.close()


📝 Explanation: Best filter for removing salt-and-pepper noise
------------------------------------------------------------

To remove salt-and-pepper noise, there are three main filters:

1. Median Filter ⭐ Best choice
   - Robust against outliers
   - Preserves edges better
   - Ideal for salt-and-pepper noise

2. Mean Filter
   - Causes image blurring
   - Does not fully remove outlier noise

3. Gaussian Filter
   - Suitable for Gaussian noise
   - Less effective for salt-and-pepper noise

Conclusion: We use the median filter to remove the noise.


Applying median filter to noisy images...
✓ Median filter applied to image 01
✓ Median filter applied to image 02
✓ Median filter applied to image 03
✓ Median filter applied to image 04
✓ Median filter applied to image 05
✓ Median filter applied to image 06
✓ Median filter applied to image 07
✓ Median filter applied to image 08
✓ Median filter applied to image 09
✓ Median filter applied to image 10
✓ Median filter applied to image 11
✓ Me

C:\Users\CD CITY\AppData\Local\Temp\ipykernel_23340\2196458171.py:140: UserWarning: Glyph 11088 (\N{WHITE MEDIUM STAR}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\CD CITY\AppData\Local\Temp\ipykernel_23340\2196458171.py:142: UserWarning: Glyph 11088 (\N{WHITE MEDIUM STAR}) missing from font(s) DejaVu Sans.
  plt.savefig(filter_comparison_path, dpi=150, bbox_inches='tight')


✓ Filter comparison image saved in e:\University_Projects\Vision\Computer-Vision-Course\HW2\outputs\filter_comparison.png
